In [1]:
from datasets import load_dataset, DownloadConfig
from itertools import islice

def load_flan_val(n=None, streaming=False, percent=1):
    dc = DownloadConfig()
    try:
        ds = load_dataset("Muennighoff/flan", split="validation", streaming=streaming, download_config=dc)
    except Exception:
        split = f"train[:{percent}%]" if not streaming else "train"
        ds = load_dataset("Muennighoff/flan", split=split, streaming=streaming, download_config=dc)
    if streaming and n:
        return list(islice(ds, n))
    if not streaming and n:
        return ds.select(range(min(n, len(ds))))
    return ds

In [2]:
ds = load_flan_val(n=5, streaming=True)
print(ds)

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

[{'inputs': 'Stewart,  Per our discussion I have changed Modesto basis option deal #520328 to reflect the restructured terms:  * original peak deal (leg 1) deal end date was changed from 9/30/01 to 8/5/01. * new leg 3 was created for 8/6/01-9/30/01. All terms stay the same except that strike price was changed to $3 and an extra premium was added for $2/MWh for the new leg. I have also put a note in the comment field requesting the confirm group to send documents to you before sending to counterparty. Please let me know if you have any questions. Thanks, \nPropose a subject line for this email?', 'targets': 'Modesto deal changes made', 'task': 'aeslc_10templates'}, {'inputs': 'Write a subject line for this message:\n\nScott,  Thanks for the help with Dave Duran. He wants to offer me a job, and he sent me to talk to Louise Kitchen today, which I did. She told me that she wanted to check my references. I was wondering if you would be willing to drop her a note on me. She would probably pu

In [3]:
# Create config_large.json by appending per-task samples from the FLAN stream
import json, os
from itertools import islice

per_task = 200  # adjust as needed
cfg_in = '../config/config2.json'
cfg_out = '../config/config_large.json'

with open(cfg_in, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

stream = load_flan_val(n=None, streaming=True)
task_streams = {}
for ex in stream:
    task_name = ex['task']
    if task_name not in task_streams:
        task_streams[task_name] = []
    task_streams[task_name].append(ex)
    
new_cfg = []
for task in cfg:
    added = []
    task_stream = task_streams.get(f"{task['model_name']}_10templates", None)
    for ex in islice(task_stream, per_task):
        inp = ex.get('input') or ex.get('inputs') or ex.get('instruction') or ex.get('context') or ex.get('prompt') or ''
        tgt = ex.get('target') or ex.get('targets') or ex.get('output') or ex.get('response') or ex.get('answer') or ''
        added.append({'inputs': inp, 'targets': tgt})
    task['sample'] = added
    new_cfg.append(task)

os.makedirs(os.path.dirname(cfg_out), exist_ok=True)
with open(cfg_out, 'w', encoding='utf-8') as f:
    json.dump(new_cfg, f, ensure_ascii=False, indent=2)

print(f'Wrote {cfg_out} with {len(new_cfg)} tasks (appended up to {per_task} samples each)')


Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Wrote ../config/config_large.json with 48 tasks (appended up to 200 samples each)
